In [0]:
file_path = "/Volumes/bootcamp_catalog/default/raw_files/orders.csv"

display(dbutils.fs.ls("/Volumes/bootcamp_catalog/default/raw_files/"))

path,name,size,modificationTime
dbfs:/Volumes/bootcamp_catalog/default/raw_files/orders.csv,orders.csv,950767,1777550212000


In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, trim, to_date, avg, sum as spark_sum, count, round as spark_round

In [0]:
schema = StructType([
    StructField("OrderID", StringType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ItemName", StringType(), True),
    StructField("Quantity", StringType(), True),
    StructField("Price", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Status", StringType(), True)
])

In [0]:
try:
    orders_df = spark.read.option("header", "true").schema(schema).csv(file_path)

    print("CSV file loaded successfully.")

except Exception as e:
    print("File loading failed.")
    print(e)

display(orders_df)

CSV file loaded successfully.


OrderID,CustomerID,ItemName,Quantity,Price,Date,Status
1,1818,Phone,4,627.1,2024-04-20,Completed
2,1771,Laptop,3,1806.55,2024-07-01,Completed
3,1733,Mouse,3,268.58,2024-11-23,Cancelled
4,1449,Headphones,1,994.15,2024-05-04,Shipped
5,1861,Monitor,3,447.73,2024-02-25,Completed
6,1230,Mouse,4,228.31,2024-04-02,Completed
7,1329,Tablet,1,821.94,2024-07-11,Cancelled
8,1571,Tablet,2,1480.11,2024-08-10,Pending
9,1185,Monitor,4,773.53,2024-09-09,Completed
10,1134,Keyboard,1,748.04,2024-07-01,Completed


In [0]:
cleaned_df = orders_df \
    .withColumn("OrderID", col("OrderID").cast("int")) \
    .withColumn("CustomerID", col("CustomerID").cast("int")) \
    .withColumn("Quantity", col("Quantity").cast("int")) \
    .withColumn("Price", col("Price").cast("double")) \
    .withColumn("Date", to_date(col("Date"), "yyyy-MM-dd")) \
    .withColumn("ItemName", trim(col("ItemName"))) \
    .withColumn("Status", trim(col("Status")))

cleaned_df = cleaned_df.fillna({
    "ItemName": "Unknown Item",
    "Status": "Unknown",
    "OrderID": 0,
    "CustomerID": 0,
    "Quantity": 0,
    "Price": 0.0
})

display(cleaned_df)

OrderID,CustomerID,ItemName,Quantity,Price,Date,Status
1,1818,Phone,4,627.1,2024-04-20,Completed
2,1771,Laptop,3,1806.55,2024-07-01,Completed
3,1733,Mouse,3,268.58,2024-11-23,Cancelled
4,1449,Headphones,1,994.15,2024-05-04,Shipped
5,1861,Monitor,3,447.73,2024-02-25,Completed
6,1230,Mouse,4,228.31,2024-04-02,Completed
7,1329,Tablet,1,821.94,2024-07-11,Cancelled
8,1571,Tablet,2,1480.11,2024-08-10,Pending
9,1185,Monitor,4,773.53,2024-09-09,Completed
10,1134,Keyboard,1,748.04,2024-07-01,Completed


In [0]:
cleaned_df.printSchema()

root
 |-- OrderID: integer (nullable = false)
 |-- CustomerID: integer (nullable = false)
 |-- ItemName: string (nullable = false)
 |-- Quantity: integer (nullable = false)
 |-- Price: double (nullable = false)
 |-- Date: date (nullable = true)
 |-- Status: string (nullable = false)



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bootcamp_catalog.raw")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS IN bootcamp_catalog").show()

+------------------+
|      databaseName|
+------------------+
|   bootcamp_schema|
|           default|
|information_schema|
|               raw|
|            silver|
+------------------+



In [0]:
cleaned_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bootcamp_catalog.raw.staging_orders_db")

print("Delta table created successfully: bootcamp_catalog.raw.staging_orders_db")

Delta table created successfully: bootcamp_catalog.raw.staging_orders_db


In [0]:
spark.sql("""
    SELECT *
    FROM bootcamp_catalog.raw.staging_orders_db
    LIMIT 10
""").show()

+-------+----------+----------+--------+-------+----------+---------+
|OrderID|CustomerID|  ItemName|Quantity|  Price|      Date|   Status|
+-------+----------+----------+--------+-------+----------+---------+
|      1|      1818|     Phone|       4|  627.1|2024-04-20|Completed|
|      2|      1771|    Laptop|       3|1806.55|2024-07-01|Completed|
|      3|      1733|     Mouse|       3| 268.58|2024-11-23|Cancelled|
|      4|      1449|Headphones|       1| 994.15|2024-05-04|  Shipped|
|      5|      1861|   Monitor|       3| 447.73|2024-02-25|Completed|
|      6|      1230|     Mouse|       4| 228.31|2024-04-02|Completed|
|      7|      1329|    Tablet|       1| 821.94|2024-07-11|Cancelled|
|      8|      1571|    Tablet|       2|1480.11|2024-08-10|  Pending|
|      9|      1185|   Monitor|       4| 773.53|2024-09-09|Completed|
|     10|      1134|  Keyboard|       1| 748.04|2024-07-01|Completed|
+-------+----------+----------+--------+-------+----------+---------+



In [0]:
display(spark.sql("""
    SELECT COUNT(*) AS total_orders
    FROM bootcamp_catalog.raw.staging_orders_db
"""))

total_orders
20000


In [0]:
source_avg_df = cleaned_df.select(
    spark_round(avg(col("Price") * col("Quantity")), 2).alias("source_avg_order_value")
)

target_avg_df = spark.sql("""
    SELECT ROUND(AVG(Price * Quantity), 2) AS target_avg_order_value
    FROM bootcamp_catalog.raw.staging_orders_db
""")

source_avg = source_avg_df.collect()[0]["source_avg_order_value"]
target_avg = target_avg_df.collect()[0]["target_avg_order_value"]

comparison_df = spark.createDataFrame(
    [(source_avg, target_avg, source_avg == target_avg)],
    ["Source_Avg_Order_Value", "Target_Avg_Order_Value", "Is_Matched"]
)

display(comparison_df)

Source_Avg_Order_Value,Target_Avg_Order_Value,Is_Matched
2561.29,2561.29,true


In [0]:
executive_summary_df = cleaned_df.select(
    count("*").alias("Total_Orders"),
    spark_round(spark_sum(col("Price") * col("Quantity")), 2).alias("Total_Revenue"),
    spark_round(avg(col("Price") * col("Quantity")), 2).alias("Average_Order_Value")
)

display(executive_summary_df)

Total_Orders,Total_Revenue,Average_Order_Value
20000,5.122579869E7,2561.29


In [0]:
customer_spend_df = cleaned_df.groupBy("CustomerID").agg(
        spark_round(spark_sum(col("Price") * col("Quantity")), 2).alias("Total_Spend")
    ).orderBy(col("Total_Spend").desc())

display(customer_spend_df)

CustomerID,Total_Spend
1702,98373.67
1504,97505.18
1478,96925.5
1085,96707.19
1371,95080.43
1833,93179.93
1332,91986.91
1800,89470.47
1193,88839.79
1033,87958.73


In [0]:
# Cleanup Script

spark.sql("DROP TABLE IF EXISTS bootcamp_catalog.raw.staging_orders_db")

print("Cleanup completed. Delta table dropped.")